In [ ]:
from google.colab import files
files.upload()  # Click on the "Choose Files" button and select your kaggle.json file


In [ ]:
# Create the .kaggle directory
!mkdir -p ~/.kaggle

# Move the kaggle.json file to the .kaggle directory
!mv kaggle.json ~/.kaggle/

# Set the permissions
!chmod 600 ~/.kaggle/kaggle.json


In [ ]:
# Install Kaggle if not already installed
!pip install kaggle

# Install other necessary libraries
!pip install pandas numpy scikit-learn joblib alibi-detect matplotlib seaborn


In [ ]:
# Download the Titanic dataset
!kaggle competitions download -c titanic

# Unzip the downloaded files
!unzip titanic.zip -d titanic_data


In [ ]:
import os

# List the files in the titanic_data directory
for dirname, _, filenames in os.walk('titanic_data'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# Load the training data
data = pd.read_csv('titanic_data/train.csv')

# Display the first few rows
data.head()


In [ ]:
# Imputation and encoding are fitted after the split below.
data = data.drop(["PassengerId", "Name", "Ticket", "Cabin"], axis=1)


In [ ]:
X = data.drop("Survived", axis=1)
y = data["Survived"]
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
numeric = ["Pclass", "Age", "SibSp", "Parch", "Fare"]
categorical = ["Sex", "Embarked"]
preprocessor = ColumnTransformer([
    ("numeric", SimpleImputer(strategy="median"), numeric),
    ("categorical", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                             ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), categorical),
])
X_train = pd.DataFrame(preprocessor.fit_transform(X_train_raw), columns=preprocessor.get_feature_names_out())
X_test = pd.DataFrame(preprocessor.transform(X_test_raw), columns=preprocessor.get_feature_names_out())


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import joblib

# Initialize the model
clf = RandomForestClassifier(n_estimators=100, random_state=42)

# Train the model
clf.fit(X_train, y_train)

# Make predictions on the test set
y_pred = clf.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy:.2f}")

# Save the trained model
joblib.dump(clf, 'model.joblib')


In [ ]:
from alibi_detect.cd import KSDrift

# Initialize the KS Drift detector with the training data as reference
detector = KSDrift(X_train.values, p_val=0.05)

# Save the detector for future use
joblib.dump(detector, 'drift_detector.joblib')


In [ ]:
def check_drift(new_data):
    if len(new_data) < 30:
        raise ValueError("Batch drift requires at least 30 rows; this is not a power guarantee.")
    detector = joblib.load("drift_detector.joblib")
    return int(detector.predict(new_data)["data"]["is_drift"])


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Simulate monitoring by evaluating on batches
batch_size = 50
accuracies = []
drifts = []

for i in range(0, len(X_test), batch_size):
    X_batch = X_test.iloc[i:i+batch_size]
    y_batch = y_test.iloc[i:i+batch_size]

    # Make predictions
    y_pred_batch = clf.predict(X_batch)

    # Calculate accuracy
    acc = accuracy_score(y_batch, y_pred_batch)
    accuracies.append(acc)

    # Check for drift
    is_drift = check_drift(X_batch.values) if len(X_batch) >= 30 else np.nan
    drifts.append(is_drift)

# Plot the accuracies
plt.figure(figsize=(12, 6))
plt.plot(accuracies, marker='o')
plt.title('Model Accuracy Over Batches')
plt.xlabel('Batch Number')
plt.ylabel('Accuracy')
plt.ylim(0, 1)
plt.grid(True)
plt.show()

# Plot drift occurrences
plt.figure(figsize=(12, 4))
plt.plot(drifts, marker='x', color='red')
plt.title('Data Drift Detection Over Batches')
plt.xlabel('Batch Number')
plt.ylabel('Drift Detected')
plt.yticks([0, 1], ['No', 'Yes'])
plt.grid(True)
plt.show()


In [ ]:
# Compare distributions of a feature (e.g., Age) before and after drift
plt.figure(figsize=(12, 6))
sns.kdeplot(X_train_raw['Age'], label='Training Age', shade=True)
sns.kdeplot(X_test_raw['Age'], label='Test Age', shade=True)
plt.title('Age Distribution: Training vs Test')
plt.xlabel('Age')
plt.ylabel('Density')
plt.legend()
plt.show()


In [ ]:
# Install ipywidgets if not already installed
!pip install ipywidgets

# Import necessary libraries
import ipywidgets as widgets
from IPython.display import display, clear_output


In [ ]:
def predict_survival(pclass, sex, age, sibsp, parch, fare, embarked):
    raw = pd.DataFrame({"Pclass": [pclass], "Sex": [sex.lower()], "Age": [age],
                        "SibSp": [sibsp], "Parch": [parch], "Fare": [fare], "Embarked": [embarked.upper()]})
    values = pd.DataFrame(preprocessor.transform(raw), columns=preprocessor.get_feature_names_out())
    return clf.predict(values)[0], None  # A single row cannot establish batch drift.


In [ ]:
# Define widgets
pclass_widget = widgets.Dropdown(
    options=[1, 2, 3],
    value=3,
    description='Pclass:',
    disabled=False,
)

sex_widget = widgets.Dropdown(
    options=['male', 'female'],
    value='male',
    description='Sex:',
    disabled=False,
)

age_widget = widgets.FloatText(
    value=30.0,
    description='Age:',
    disabled=False
)

sibsp_widget = widgets.IntSlider(
    value=0,
    min=0,
    max=10,
    step=1,
    description='SibSp:',
    disabled=False,
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='d'
)

parch_widget = widgets.IntSlider(
    value=0,
    min=0,
    max=10,
    step=1,
    description='Parch:',
    disabled=False,
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='d'
)

fare_widget = widgets.FloatText(
    value=32.2,
    description='Fare:',
    disabled=False
)

embarked_widget = widgets.Dropdown(
    options=['C', 'Q', 'S'],
    value='S',
    description='Embarked:',
    disabled=False,
)

button = widgets.Button(
    description='Predict',
    disabled=False,
    button_style='success',
    tooltip='Click to predict',
    icon='check'
)

output = widgets.Output()


In [ ]:
def on_button_clicked(b):
    with output:
        clear_output()
        # Get input values
        pclass = pclass_widget.value
        sex = sex_widget.value
        age = age_widget.value
        sibsp = sibsp_widget.value
        parch = parch_widget.value
        fare = fare_widget.value
        embarked = embarked_widget.value

        # Make prediction
        prediction, is_drift = predict_survival(pclass, sex, age, sibsp, parch, fare, embarked)

        # Display results
        print(f"Survived: {'Yes' if prediction == 1 else 'No'}")
        print("Batch drift: not evaluated for a single prediction")

# Attach the event handler to the button
button.on_click(on_button_clicked)


In [ ]:
# Arrange widgets in a vertical box
ui = widgets.VBox([
    pclass_widget,
    sex_widget,
    age_widget,
    sibsp_widget,
    parch_widget,
    fare_widget,
    embarked_widget,
    button,
    output
])

# Display the interface
display(ui)
